In [23]:
import pandas as pd

In [24]:
df_joint = pd.read_csv("/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/QA/analysis/task3_cot_analysis/outputs/task3_stepwise_cot_joint_step_em_by_final_em.csv")
df_merginal = pd.read_csv("/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/QA/analysis/task3_cot_analysis/outputs/task3_stepwise_cot_marginal_step_em_by_final_em.csv")

In [25]:
df_joint

,model,prompt_variant,final_exact_match,step1_fragment_EM,step2_fragment_EM,count,frac_within_final_em
0,gemini-3-flash,single_step,0,0,0,60,0.652174
1,gemini-3-flash,single_step,0,1,0,31,0.336957
2,gemini-3-flash,single_step,0,1,1,1,0.010870
3,gemini-3-flash,single_step,1,1,0,2,0.250000
4,gemini-3-flash,single_step,1,1,1,6,0.750000
5,gemini-3.1-flash-lite,single_step,0,0,0,63,0.677419
6,gemini-3.1-flash-lite,single_step,0,1,0,30,0.322581
7,gemini-3.1-flash-lite,single_step,1,0,0,1,0.142857
8,gemini-3.1-flash-lite,single_step,1,1,0,3,0.428571
9,gemini-3.1-flash-lite,single_step,1,1,1,3,0.428571


In [26]:
df_merginal

,model,prompt_variant,final_exact_match,step,fragment_EM,count,frac_within_final_em
0,gemini-3-flash,single_step,0,step1,0,60,0.652174
1,gemini-3-flash,single_step,0,step1,1,32,0.347826
2,gemini-3-flash,single_step,0,step2,0,91,0.989130
3,gemini-3-flash,single_step,0,step2,1,1,0.010870
4,gemini-3-flash,single_step,1,step1,0,0,0.000000
...,...,...,...,...,...,...,...
67,gpt-5.2,multi_step,0,step2,1,0,0.000000
68,gpt-5.2,multi_step,1,step1,0,0,0.000000
69,gpt-5.2,multi_step,1,step1,1,0,0.000000
70,gpt-5.2,multi_step,1,step2,0,0,0.000000


In [27]:
# 모델 구분 없이 count 합산 → frac_within_final_em 재계산
# (1) prompt_variant(single_step / multi_step)는 유지
_g_joint = ["prompt_variant", "final_exact_match", "step1_fragment_EM", "step2_fragment_EM"]
df_joint_pooled = df_joint.groupby(_g_joint, as_index=False)["count"].sum()
_tot_j = df_joint_pooled.groupby(["prompt_variant", "final_exact_match"])["count"].transform("sum")
df_joint_pooled["frac_within_final_em"] = df_joint_pooled["count"] / _tot_j

_g_marg = ["prompt_variant", "final_exact_match", "step", "fragment_EM"]
df_merginal_pooled = df_merginal.groupby(_g_marg, as_index=False)["count"].sum()
# marginal: (prompt, final_em, step) 안에서 fragment_EM 0/1 비율
_tot_m = df_merginal_pooled.groupby(["prompt_variant", "final_exact_match", "step"])["count"].transform("sum")
df_merginal_pooled["frac_within_final_em"] = df_merginal_pooled["count"] / _tot_m

# (2) prompt_variant까지 합친 전체 풀 (모든 모델·프롬프트를 한 덩어리)
_g_joint_all = ["final_exact_match", "step1_fragment_EM", "step2_fragment_EM"]
df_joint_pooled_all = df_joint.groupby(_g_joint_all, as_index=False)["count"].sum()
_tot_ja = df_joint_pooled_all.groupby("final_exact_match")["count"].transform("sum")
df_joint_pooled_all["frac_within_final_em"] = df_joint_pooled_all["count"] / _tot_ja

_g_marg_all = ["final_exact_match", "step", "fragment_EM"]
df_merginal_pooled_all = df_merginal.groupby(_g_marg_all, as_index=False)["count"].sum()
_tot_ma = df_merginal_pooled_all.groupby(["final_exact_match", "step"])["count"].transform("sum")
df_merginal_pooled_all["frac_within_final_em"] = df_merginal_pooled_all["count"] / _tot_ma

In [28]:
df_joint_pooled

,prompt_variant,final_exact_match,step1_fragment_EM,step2_fragment_EM,count,frac_within_final_em
0,multi_step,0,0,0,393,0.982500
1,multi_step,0,1,0,7,0.017500
2,single_step,0,0,0,344,0.718163
3,single_step,0,1,0,132,0.275574
4,single_step,0,1,1,3,0.006263
5,single_step,1,0,0,1,0.047619
6,single_step,1,1,0,7,0.333333
7,single_step,1,1,1,13,0.619048


In [29]:
import numpy as np

df_task12_vs_cot_summary_by_model = pd.read_csv(
    "/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/QA/analysis/task3_cot_analysis/outputs/task12_vs_cot_summary.csv"
)

# model 구분 없이 통합: status==ok 만, n_paired 가중 평균 (합계 컬럼은 sum)
_ok = df_task12_vs_cot_summary_by_model[df_task12_vs_cot_summary_by_model["status"] == "ok"].copy()
_mean_cols = [
    c
    for c in _ok.columns
    if c.startswith("mean_") or c.startswith("agreement_") or c == "meta_mismatch_rate"
]


def _pool_one_group(g: pd.DataFrame) -> dict:
    w = g["n_paired"].astype(float)
    denom = w.sum()
    d = {
        "prompt_variant": g["prompt_variant"].iloc[0],
        "n_models": len(g),
        "n_paired": int(denom),
        "n_task1": int(g["n_task1"].sum()),
        "n_task2": int(g["n_task2"].sum()),
        "n_cot": int(g["n_cot"].sum()),
    }
    for c in _mean_cols:
        if c not in g.columns:
            continue
        vals = g[c].astype(float)
        mask = vals.notna() & w.gt(0)
        if mask.any() and float(w[mask].sum()) > 0:
            d[c] = float(np.average(vals[mask], weights=w[mask]))
        else:
            d[c] = np.nan
    return d


_rows = []
for _, g in _ok.groupby("prompt_variant", sort=False):
    _rows.append(_pool_one_group(g))
df_task12_vs_cot_summary = pd.DataFrame(_rows)

# single + multi 까지 한 줄로 (전체 풀)
if not _ok.empty:
    _all = _pool_one_group(_ok)
    _all["prompt_variant"] = "all (pooled)"
    df_task12_vs_cot_summary_pooled_all = pd.DataFrame([_all])
else:
    df_task12_vs_cot_summary_pooled_all = pd.DataFrame()

In [30]:
# df_task12_vs_cot_summary: model 통합 (prompt_variant별). 원본은 df_task12_vs_cot_summary_by_model
print(df_task12_vs_cot_summary.columns)
_base = ["prompt_variant", "n_models", "n_paired"]
task1_vs = _base + ['mean_task1_EM', 'mean_cot_step1_EM', 'mean_delta_step1_cot_minus_task1', 'agreement_task1_vs_cot_step1']
task2_vs = _base + ['mean_task2_EM', 'mean_cot_step2_EM', 'mean_delta_step2_cot_minus_task2', 'agreement_task2_vs_cot_step2']
df_task1_vs = df_task12_vs_cot_summary[task1_vs]
df_task2_vs = df_task12_vs_cot_summary[task2_vs]
df_task12_vs_cot_summary

Index(['prompt_variant', 'n_models', 'n_paired', 'n_task1', 'n_task2', 'n_cot',
       'mean_task1_EM', 'mean_cot_step1_EM',
       'mean_delta_step1_cot_minus_task1', 'agreement_task1_vs_cot_step1',
       'mean_task2_EM', 'mean_cot_step2_EM',
       'mean_delta_step2_cot_minus_task2', 'agreement_task2_vs_cot_step2',
       'mean_cot_final_EM', 'meta_mismatch_rate'],
      dtype='object')


,prompt_variant,n_models,n_paired,n_task1,n_task2,n_cot,mean_task1_EM,mean_cot_step1_EM,mean_delta_step1_cot_minus_task1,agreement_task1_vs_cot_step1,mean_task2_EM,mean_cot_step2_EM,mean_delta_step2_cot_minus_task2,agreement_task2_vs_cot_step2,mean_cot_final_EM,meta_mismatch_rate
0,multi_step,3,285,300,300,300,0.021053,0.014035,-0.007018,0.985965,0.000000,0.000000,0.000000,1.000000,0.000000,0.0
1,single_step,4,376,400,400,400,0.388298,0.375000,-0.013298,0.768617,0.114362,0.042553,-0.071809,0.890957,0.055851,0.0


In [31]:
df_task1_vs

,prompt_variant,n_models,n_paired,mean_task1_EM,mean_cot_step1_EM,mean_delta_step1_cot_minus_task1,agreement_task1_vs_cot_step1
0,multi_step,3,285,0.021053,0.014035,-0.007018,0.985965
1,single_step,4,376,0.388298,0.375000,-0.013298,0.768617


In [33]:
df_task2_vs

,prompt_variant,n_models,n_paired,mean_task2_EM,mean_cot_step2_EM,mean_delta_step2_cot_minus_task2,agreement_task2_vs_cot_step2
0,multi_step,3,285,0.000000,0.000000,0.000000,1.000000
1,single_step,4,376,0.114362,0.042553,-0.071809,0.890957


In [ ]:
import numpy as np  # 셀 단독 실행 시

# task3 vs task3_stepwise_cot 최종 EM (model 통합)
df_task3_vs_cot_by_model = pd.read_csv(
    "/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/QA/analysis/task3_cot_analysis/outputs/task3_vs_task3_cot_summary.csv"
)

_ok3 = df_task3_vs_cot_by_model[df_task3_vs_cot_by_model["status"] == "ok"].copy()
_mean_cols_t3 = [
    c
    for c in _ok3.columns
    if c.startswith("mean_") or c.startswith("agreement_") or c == "meta_mismatch_rate"
]
_count_cols_t3 = [
    "count_both_em1",
    "count_both_em0",
    "count_task3_em1_cot_em0",
    "count_task3_em0_cot_em1",
]


def _pool_task3_group(g) -> dict:
    w = g["n_paired"].astype(float)
    denom = w.sum()
    d = {
        "prompt_variant": g["prompt_variant"].iloc[0],
        "n_models": len(g),
        "n_paired": int(denom),
        "n_task3": int(g["n_task3"].sum()),
        "n_cot": int(g["n_cot"].sum()),
    }
    for c in _mean_cols_t3:
        if c not in g.columns:
            continue
        vals = g[c].astype(float)
        mask = vals.notna() & w.gt(0)
        if mask.any() and float(w[mask].sum()) > 0:
            d[c] = float(np.average(vals[mask], weights=w[mask]))
        else:
            d[c] = np.nan
    # 샘플 수 합으로 나눈 비율 (모델별 count 합 / n_paired 합) = 매크로 평균 비율
    for c in _count_cols_t3:
        if c in g.columns:
            d[c] = float(g[c].sum()) / denom if denom else np.nan
    return d


_rows3 = []
for _, g in _ok3.groupby("prompt_variant", sort=False):
    _rows3.append(_pool_task3_group(g))
df_task3_vs_cot = pd.DataFrame(_rows3)

if not _ok3.empty:
    _all3 = _pool_task3_group(_ok3)
    _all3["prompt_variant"] = "all (pooled)"
    df_task3_vs_cot_pooled_all = pd.DataFrame([_all3])
else:
    df_task3_vs_cot_pooled_all = pd.DataFrame()

In [ ]:
# prompt_variant별 통합 / 전체 풀
df_task3_vs_cot

In [ ]:
df_task3_vs_cot_pooled_all

In [32]:
# single_step + multi_step 까지 모두 합친 전체 풀 (한 줄)
df_task12_vs_cot_summary_pooled_all

,prompt_variant,n_models,n_paired,n_task1,n_task2,n_cot,mean_task1_EM,mean_cot_step1_EM,mean_delta_step1_cot_minus_task1,agreement_task1_vs_cot_step1,mean_task2_EM,mean_cot_step2_EM,mean_delta_step2_cot_minus_task2,agreement_task2_vs_cot_step2,mean_cot_final_EM,meta_mismatch_rate
0,all (pooled),7,661,700,700,700,0.229955,0.219365,-0.01059,0.86233,0.065053,0.024206,-0.040847,0.937973,0.03177,0.0


In [34]:
df_task3_vs_cot = pd.read_csv("/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/QA/analysis/task3_cot_analysis/outputs/task3_vs_task3_cot_summary.csv")

In [35]:
df_task3_vs_cot

,model,prompt_variant,status,n_paired,n_task3,n_cot,mean_task3_exact_match,mean_cot_exact_match,mean_delta_cot_minus_task3,agreement_exact_match_01,...,count_task3_em1_cot_em0,count_task3_em0_cot_em1,mean_task3_bleu,mean_cot_bleu,mean_delta_bleu_cot_minus_task3,mean_task3_validity,mean_cot_validity,meta_mismatch_rate,task3_path,cot_path
0,gemini-3-flash,single_step,ok,100,100,100,0.21,0.08,-0.13,0.79,...,17,4,0.960923,0.859100,-0.101823,0.99,0.89,0.0,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...
1,gemini-3.1-flash-lite,single_step,ok,100,100,100,0.04,0.07,0.03,0.93,...,2,5,0.946796,0.933562,-0.013234,0.94,0.97,0.0,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...
2,gemini-3.1-flash,single_step,ok,100,100,100,0.00,0.00,0.00,1.00,...,0,0,0.000000,0.000000,0.000000,0.00,0.00,0.0,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...
3,gpt-4o,single_step,ok,100,100,100,0.05,0.03,-0.02,0.94,...,4,2,0.933619,0.897429,-0.036190,0.89,0.75,0.0,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...
4,gpt-5.2,single_step,ok,100,100,100,0.04,0.03,-0.01,0.97,...,2,1,0.932172,0.944989,0.012818,0.90,0.93,0.0,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...
5,gemini-3.1-flash-lite,multi_step,ok,100,100,100,0.01,0.00,-0.01,0.99,...,1,0,0.910230,0.900468,-0.009762,0.90,0.97,0.0,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...
6,gemini-3.1-flash,multi_step,ok,100,100,100,0.00,0.00,0.00,1.00,...,0,0,0.000000,0.000000,0.000000,0.00,0.00,0.0,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...
7,gpt-4o,multi_step,ok,100,100,100,0.00,0.00,0.00,1.00,...,0,0,0.886238,0.873746,-0.012492,0.90,0.78,0.0,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...
8,gpt-5.2,multi_step,ok,100,100,100,0.01,0.00,-0.01,0.99,...,1,0,0.887344,0.895203,0.007860,0.88,0.94,0.0,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...,/ssd1/jueon/wj/detoxicity_model/ace_safe_ver/Q...
